In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("marquis03/cats-and-dogs")

print("Path to dataset files:", path)

100%|██████████| 9.75M/9.75M [00:30<00:00, 337kB/s]

Extracting files...


Path to dataset files: C:\Users\ACER\.cache\kagglehub\datasets\marquis03\cats-and-dogs\versions\2


'cp' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import pandas as pd
from PIL import Image
import copy
from pathlib import Path

# --- BASIC SETUP ---
IMG_SIZE = 150
BATCH_SIZE = 32
NUM_WORKERS = 2
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"
print(f"Running on device: {DEVICE}")

# Find root that contains CSV labels and image folders
candidate_roots = [Path.cwd(), Path.cwd() / "Lab6", Path.cwd().parent]
DATA_ROOT = None
for root in candidate_roots:
    if (root / "train.csv").exists() and (root / "val.csv").exists():
        DATA_ROOT = root
        break

if DATA_ROOT is None:
    raise FileNotFoundError(
        "Could not find train.csv and val.csv. "
        "Open notebook in the correct workspace or set DATA_ROOT manually."
    )

TRAIN_CSV = DATA_ROOT / "train.csv"
VAL_CSV = DATA_ROOT / "val.csv"
print(f"Data root: {DATA_ROOT.resolve()}")
print(f"Train CSV: {TRAIN_CSV}")
print(f"Val CSV: {VAL_CSV}")

# --- 1. DATA AUGMENTATION ---
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_transforms = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ]
)

val_transforms = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ]
)


class CsvImageDataset(Dataset):
    def __init__(self, csv_path: Path, root_dir: Path, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.df = pd.read_csv(csv_path)

        if "image:FILE" not in self.df.columns or "category" not in self.df.columns:
            raise ValueError(
                "CSV must contain columns: 'image:FILE' and 'category'. "
                f"Found: {list(self.df.columns)}"
            )

        self.image_paths = [
            self.root_dir / str(rel_path).replace("\\", "/")
            for rel_path in self.df["image:FILE"].tolist()
        ]
        self.labels = [int(x) for x in self.df["category"].tolist()]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]

        image = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            image = self.transform(image)

        return image, label


train_data = CsvImageDataset(TRAIN_CSV, DATA_ROOT, transform=train_transforms)
val_data = CsvImageDataset(VAL_CSV, DATA_ROOT, transform=val_transforms)

print(f"Train size: {len(train_data)} | Val size: {len(val_data)}")

train_loader = DataLoader(
    train_data,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    val_data,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

# --- 3. CLASS BALANCING ---
train_size = len(train_data)
val_size = len(val_data)

labels = train_data.labels
num_class_0 = labels.count(0)
num_class_1 = labels.count(1)

if num_class_1 == 0:
    raise ValueError("No class-1 samples in train set, cannot compute pos_weight")

pos_weight = torch.tensor([num_class_0 / num_class_1], dtype=torch.float32).to(DEVICE)
print(f"Class count - class 0: {num_class_0}, class 1: {num_class_1}")
print(f"Positive weight: {pos_weight.item():.4f}")


# --- 4. PURE CNN ARCHITECTURE ---
class PureCNN(nn.Module):
    def __init__(self):
        super(PureCNN, self).__init__()

        # Block 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2, 2)

        # Block 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2, 2)

        # Block 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)

        # Block 4
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.pool4 = nn.MaxPool2d(2, 2)

        self.relu = nn.ReLU()

        # Size after 4 pooling layers for 150x150 input is 9x9
        self.flatten_dim = 256 * 9 * 9

        # Classifier
        self.fc1 = nn.Linear(self.flatten_dim, 512)
        self.bn_fc = nn.BatchNorm1d(512)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(512, 1)

    def forward(self, x):
        x = self.pool1(self.relu(self.bn1(self.conv1(x))))
        x = self.pool2(self.relu(self.bn2(self.conv2(x))))
        x = self.pool3(self.relu(self.bn3(self.conv3(x))))
        x = self.pool4(self.relu(self.bn4(self.conv4(x))))

        x = x.view(-1, self.flatten_dim)

        x = self.relu(self.bn_fc(self.fc1(x)))
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = PureCNN().to(DEVICE)

# --- 5. LOSS, OPTIMIZER, SCHEDULER ---
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-3)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=4, min_lr=1e-6, verbose=True
)

# --- 6. TRAIN LOOP + EARLY STOPPING ---
epochs = 50
patience = 12
best_val_acc = 0.0
epochs_no_improve = 0
best_model_weights = copy.deepcopy(model.state_dict())


def get_accuracy(outputs, labels):
    preds = (torch.sigmoid(outputs) >= 0.5).float()
    correct = (preds == labels).sum().item()
    return correct


for epoch in range(epochs):
    model.train()
    running_loss, running_corrects = 0.0, 0

    for inputs, labels in train_loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        labels = labels.float().unsqueeze(1).to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_corrects += get_accuracy(outputs, labels)

    train_loss = running_loss / train_size
    train_acc = running_corrects / train_size

    model.eval()
    val_loss, val_corrects = 0.0, 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(DEVICE, non_blocking=True)
            labels = labels.float().unsqueeze(1).to(DEVICE, non_blocking=True)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            val_corrects += get_accuracy(outputs, labels)

    val_loss = val_loss / val_size
    val_acc = val_corrects / val_size

    print(
        f"Epoch {epoch + 1:02d}/{epochs} | "
        f"Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}"
    )

    scheduler.step(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_weights = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(
                f"\n=> Early stopping at epoch {epoch + 1}. "
                f"No improvement after {patience} epochs."
            )
            break

model.load_state_dict(best_model_weights)
torch.save(model.state_dict(), "best_pure_cnn_pytorch.pth")
print(f"Training complete! Saved best model with accuracy: {best_val_acc:.4f}")

Dang chay tren thiet bi: cuda
Data root: C:\Users\ACER\Desktop\Phat_\gioithieu_hocsau\Lab6
Classes: ['cat', 'dog']
Train size: 275 | Val size: 70
So luong - Class 0: 95, Class 1: 180
Positive weight: 0.5278


c:\Python39\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


RuntimeError: DataLoader worker (pid(s) 2136, 9800) exited unexpectedly